ERP Analysis Procedure**

### **1. Load Raw EEG File**

* Load the `.set` EEG file for a participant using MNE.

---

### **2. Re-reference**

* Re-reference EEG data to the **average of LM (Left Mastoid)** and **RM (Right Mastoid)** **before any other step**.

---

### **3. ICA for Blink Artifact Removal**

* Run **ICA on the unfiltered or minimally filtered data** (preferably **raw or lightly filtered**, e.g., 0.01–50 Hz).
* Use **all 128 channels** (not just 81).
* **Plot components** to visually inspect and **identify blink-related components**.

  * These are typically **among the first 5** due to high amplitude.
* **Manually exclude blink components only** (avoid over-correction).

---

### **4. Apply ICA**

* Apply ICA solution to the raw data.
* Visually **inspect ICA-corrected signal** (before filtering).

---

### **5. Interpolate Bad Channels**

* Identify bad/noisy channels (especially **central anterior/medial** regions).
* **Interpolate them using weighted interpolation**.
* Try to limit exclusions; preserve as much spatial coverage as possible.

---

### **6. Filter**

* Apply band-pass filter **from 0.01 Hz to 50 Hz**.

  * Can be done before or after ICA depending on blink removal performance.
* Apply **notch filter at 60 Hz** to remove line noise.

---

### **7. Extract Events**

* Extract events from annotations:

  * `Wd2E` = normal context
  * `Wd2N` = slowed context

---

### **8. Create Epochs**

* Epochs should be **time-locked to event markers** (`Wd2E`/`Wd2N`).
* Use **epoch window**: -100 ms to 800 ms.
* Apply **baseline correction** using the **100 ms pre-onset period**.

---

### **9. Reject Artifactual Epochs**

* Automatically or manually reject epochs with high-amplitude noise or eye movement remnants.
* Use voltage thresholds, peak-to-peak rejection, or visual inspection.

---

### **10. Save Cleaned Epochs**

* Save the cleaned and processed epochs per subject (`.fif` format).

---

### **11. Compute Subject-level ERPs**

* For each subject, compute average ERPs:

  * ERP for all `Wd2E` epochs (normal context)
  * ERP for all `Wd2N` epochs (slowed context)

---

### **12. Grand Average (Group-Level ERPs)**

* Average across all 21 participants:

  * Grand average ERP for `Wd2E`
  * Grand average ERP for `Wd2N`

---

### **13. Plot ERPs (9 Scalp Regions)**

* Use all **128 channels** and group into 9 regions:

  * Left/Medial/Right × Anterior/Central/Posterior
* Plot ERP waveforms over time (−100 ms to 400 ms) showing both conditions.
* Highlight **N100** and **N280** components.




In [ ]:
# SETUP
from google.colab import drive
drive.mount('/content/drive')

!pip install mne --quiet
import mne
import os
import gc
import numpy as np
from scipy.signal import hilbert
import matplotlib.pyplot as plt

In [ ]:
# Set path
root_dir = '/content/drive/MyDrive/Distal_Rate_ERP_Summer2025/Data/SpeechRate2025/EEGData'
participants = sorted([p for p in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, p))])

def preprocess_and_save_with_ica(participant):
    participant_path = os.path.join(root_dir, participant)
    set_files = [f for f in os.listdir(participant_path) if f.endswith('.set')]

    if not set_files:
        print(f"No .set files found for {participant}")
        return None

    for set_file in set_files:
        set_name = os.path.splitext(set_file)[0]
        raw_path = os.path.join(participant_path, set_file)
        fif_path = os.path.join(participant_path, f'{set_name}_clean_raw.fif')
        ica_path = os.path.join(participant_path, f'{set_name}-ica.fif')
        ica_fig_path = os.path.join(participant_path, f"{set_name}_ica_components.png")

        # Skip if already processed
        if os.path.exists(fif_path) and os.path.exists(ica_path) and os.path.exists(ica_fig_path):
            print(f"Already preprocessed: {set_file}")
            continue

        try:
            raw = mne.io.read_raw_eeglab(raw_path, preload=True)

            # Re-reference to LM/RM (E57, E100)
            try:
                lm = raw.copy().pick(['E57']).get_data()
                rm = raw.copy().pick(['E100']).get_data()
                mastoid_avg = (lm + rm) / 2
                raw._data = raw.get_data() - mastoid_avg
                print(f"{set_file}: Re-referenced using LM/RM average.")
            except Exception as e:
                print(f"{set_file}: Error during referencing — {e}")
                continue

            # Run ICA
            ica = mne.preprocessing.ICA(n_components=20, random_state=97, max_iter='auto')
            ica.fit(raw)
            print(f"{set_file}: ICA fitted.")

            # Save ICA solution
            ica.save(ica_path, overwrite=True)
            print(f"{set_file}: ICA saved to {ica_path}")

            # Plot & save ICA topomap
            fig = ica.plot_components(inst=raw, show=False)
            fig.savefig(ica_fig_path, dpi=150)
            print(f"{set_file}: ICA plot saved to {ica_fig_path}")

            # Save re-referenced raw
            raw.save(fif_path, overwrite=True)
            print(f"{set_file}: Raw saved to {fif_path}")

            # Free memory
            del raw, ica, fig
            gc.collect()

        except Exception as e:
            print(f"{set_file}: Error during processing — {e}")
            continue

# Run all participants
for pid in participants:
    preprocess_and_save_with_ica(pid)
